# Lab 14: clasificador de latas Monster / Red Bull

**Integrantes:** Mathias Carrera, Bastian Contreras

**Clases:**
- `monster`: lata de Monster Energy, de cualquier sabor y color.
- `redbull`: lata de Red Bull, de cualquier sabor y color.
- `ninguna`: no hay ninguna de las dos (vaso de agua, otras bebidas, objetos, fondos sin lata).

**Por qué:** queremos un modelo que reconozca las dos marcas y además sepa decir cuándo no hay ninguna, no que fuerce una de las dos. La clase `ninguna` incluye objetos parecidos (vasos, botellas, otras latas) para que no se limite a detectar "una lata".

**Criterio de etiquetado:** la clase es la marca de la lata visible en la foto. Si hay una lata de cada una, no se usa la foto. Si la marca no se distingue, tampoco. Todas las fotos son propias, tomadas con celular y reducidas a 512 px.

## Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torchmetrics
import torchvision
from lightning.pytorch import LightningModule, Trainer, seed_everything
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder

seed_everything(42)

## Datos

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Dos vistas de data/train: una con aumento de datos (entrenar) y otra sin (validar)
train_full = ImageFolder("data/train", transform=train_transform)
val_full = ImageFolder("data/train", transform=eval_transform)
test_ds = ImageFolder("data/test", transform=eval_transform)

CLASS_NAMES = train_full.classes
NUM_CLASSES = len(CLASS_NAMES)

# 80 % entrenamiento / 20 % validación, con semilla fija
generator = torch.Generator().manual_seed(42)
indices = torch.randperm(len(train_full), generator=generator).tolist()
n_val = int(0.2 * len(indices))
train_ds = Subset(train_full, indices[n_val:])
val_ds = Subset(val_full, indices[:n_val])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)
test_loader = DataLoader(test_ds, batch_size=32)

print(CLASS_NAMES)
for split in ("train", "test"):
    counts = {c: len(list(Path(f"data/{split}/{c}").glob("*"))) for c in CLASS_NAMES}
    print(split, counts)

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(12, 8))
for row, class_index in enumerate(range(NUM_CLASSES)):
    samples = [i for i, (_, y) in enumerate(val_full.samples) if y == class_index][:5]
    for axis, i in zip(axes[row], samples):
        axis.imshow(plt.imread(val_full.samples[i][0]))
        axis.set_title(CLASS_NAMES[class_index])
        axis.axis("off")
plt.tight_layout()
plt.show()

## Modelo: ResNet-18 pre-entrenada con fine-tuning

In [ ]:
class CanModel(LightningModule):
    def __init__(self, num_classes, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
        self.model.fc = nn.Linear(512, num_classes)
        self.val_acc = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.test_acc = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.confmat = torchmetrics.ConfusionMatrix(task="multiclass", num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_nb):
        x, y = batch
        loss = F.cross_entropy(self(x), y)
        self.log("train_loss", loss, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_nb):
        x, y = batch
        out = self(x)
        self.val_acc(out, y)
        self.log("val_acc", self.val_acc, on_epoch=True, prog_bar=True)
        self.log("val_loss", F.cross_entropy(out, y), prog_bar=True)

    def test_step(self, batch, batch_nb):
        x, y = batch
        out = self(x)
        self.test_acc(out, y)
        self.confmat.update(out, y)
        self.log("test_acc", self.test_acc, on_epoch=True, prog_bar=True)
        self.log("test_loss", F.cross_entropy(out, y), prog_bar=True)

    def on_test_epoch_end(self):
        cm = self.confmat.compute()
        fig, ax = plt.subplots(figsize=(6, 5))
        im = ax.imshow(cm.cpu().numpy(), cmap="Blues")
        ax.set_xticks(range(len(CLASS_NAMES)))
        ax.set_yticks(range(len(CLASS_NAMES)))
        ax.set_xticklabels(CLASS_NAMES)
        ax.set_yticklabels(CLASS_NAMES)
        ax.set_xlabel("Predicción")
        ax.set_ylabel("Real")
        for i in range(len(CLASS_NAMES)):
            for j in range(len(CLASS_NAMES)):
                ax.text(j, i, int(cm[i, j]), ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
        ax.set_title("Matriz de confusión (test)")
        fig.colorbar(im)
        plt.show()
        print(cm)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

## Entrenamiento

In [ ]:
model = CanModel(NUM_CLASSES)
checkpoint = ModelCheckpoint(dirpath="log/checkpoints", save_top_k=1, monitor="val_loss", mode="min")

trainer = Trainer(
    accelerator="auto",
    max_epochs=10,
    logger=TensorBoardLogger("log"),
    callbacks=[checkpoint],
)
trainer.fit(model, train_loader, val_loader)

## Evaluación en test

In [ ]:
trainer.test(model, dataloaders=test_loader, ckpt_path="best")

## Predicciones sobre el test

In [ ]:
model.eval()
paths = [p for p, _ in test_ds.samples]
shown = list(range(0, len(paths), max(1, len(paths) // 10)))[:10]

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for axis, i in zip(axes.flatten(), shown):
    x, y = test_ds[i]
    with torch.no_grad():
        probs = F.softmax(model(x.unsqueeze(0).to(model.device)), dim=1)[0]
    pred = probs.argmax().item()
    axis.imshow(plt.imread(paths[i]))
    axis.set_title(f"pred {CLASS_NAMES[pred]} ({probs[pred]:.0%})\nreal {CLASS_NAMES[y]}",
                   color="tab:green" if pred == y else "tab:red", fontsize=9)
    axis.axis("off")
plt.tight_layout()
plt.show()